# Water Quality — Real Machine Learning Training & Evaluation
**Reproducible extension to the team’s unified EDA/preprocessing notebook.**

This notebook trains five models and selects a winner **only from the training-set cross-validation**. The held-out test set is evaluated only once, after model selection. **Do not interpret dataset Potability predictions as safe-to-drink decisions.**

This notebook uses shared functions from `ml/train.py` so CLI, website, and notebook use the *same* feature schema and experimental method instead of duplicating code.

## 0. Environment and paths
Run from the repository root or `notebooks/` folder. Install packages from `requirements.txt`. Use local original CSV, not a remotely fetched dataset.

In [1]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
project_root=Path.cwd().resolve()
if project_root.name=='notebooks': project_root=project_root.parent
assert (project_root/'data'/'water_potability.csv').exists(), 'Run from repository root or notebooks/ folder'
sys.path.insert(0,str(project_root))
from ml.train import (FEATURES, TARGET, SEED, load_data, make_split, build_candidates, benchmark_models, evaluate_model, save_outputs)
from ml.predict import predict_one, artifacts
print('Project root:',project_root)

Project root: /mnt/data/Water_Quality_ML_First


## 1. Load and verify the raw dataset
Exactly nine numeric features plus the binary `Potability` label. Missingness is retained until preprocessing (to avoid leakage).

In [2]:
df=load_data()
print('Shape:', df.shape, '| Duplicates:', df.duplicated().sum())
print('Class counts:', df[TARGET].value_counts().to_dict())
display(df.isna().sum().to_frame('Missing'))
display(df.head(3))

Shape: (3276, 10) | Duplicates: 0
Class counts: {0: 1998, 1: 1278}


,Missing
ph,491
Hardness,0
Solids,0
Chloramines,0
Sulfate,781
Conductivity,0
Organic_carbon,0
Trihalomethanes,162
Turbidity,0
Potability,0


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0


## 2. Make the held-out 80/20 stratified split (before preprocessing)
All imputers and scalers are fitted **inside each model Pipeline**, within each training CV fold. The 20% hold-out is not touched during model comparison.

In [3]:
X_train,X_test,y_train,y_test=make_split(df)
assert not set(X_train.index) & set(X_test.index)
print('Train:',X_train.shape, y_train.value_counts(normalize=True).round(3).to_dict())
print('Held-out test:',X_test.shape,y_test.value_counts(normalize=True).round(3).to_dict())

Train: (2620, 9) {0: 0.61, 1: 0.39}
Held-out test: (656, 9) {0: 0.61, 1: 0.39}


## 3. Build five model Pipelines
`SimpleImputer(median) → StandardScaler → Classifier` is refitted separately in every CV fold. Scaling is unnecessary for Random Forest but kept for a single consistent feature contract.

In [4]:
candidates=build_candidates()
for name,pipe in candidates.items(): print(name, '→',list(pipe.named_steps))

DummyClassifier → ['imputer', 'scaler', 'classifier']
Logistic Regression → ['imputer', 'scaler', 'classifier']
K-Nearest Neighbors → ['imputer', 'scaler', 'classifier']
Support Vector Machine → ['imputer', 'scaler', 'classifier']
Random Forest → ['imputer', 'scaler', 'classifier']


## 4. Compare via five-fold training-only cross-validation
The metrics include balanced accuracy, macro-F1, per-class recall, ROC-AUC, and **false-safe rate** (actual 0 predicted 1). Candidate selection criterion is the largest mean CV balanced accuracy, with macro F1 and lower false-safe rate tie-breakers.

In [5]:
comparison=benchmark_models(X_train,y_train)
columns=['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean','cv_recall_not_potable_mean','cv_recall_potable_mean','cv_false_safe_rate_mean','cv_roc_auc_mean']
display(comparison[columns].round(4))

  DummyClassifier           balanced accuracy=0.500  false-safe rate=0.000


  Logistic Regression       balanced accuracy=0.500  false-safe rate=0.000


  K-Nearest Neighbors       balanced accuracy=0.575  false-safe rate=0.127


  Support Vector Machine    balanced accuracy=0.610  false-safe rate=0.073


  Random Forest             balanced accuracy=0.617  false-safe rate=0.102


,model,cv_accuracy_mean,cv_balanced_accuracy_mean,cv_macro_f1_mean,cv_recall_not_potable_mean,cv_recall_potable_mean,cv_false_safe_rate_mean,cv_roc_auc_mean
0,Random Forest,0.6786,0.6168,0.6103,0.8980,0.3356,0.1020,0.6830
1,Support Vector Machine,0.6798,0.6102,0.5977,0.9268,0.2935,0.0732,0.7019
2,K-Nearest Neighbors,0.6408,0.5754,0.5615,0.8730,0.2779,0.1270,0.6544
3,DummyClassifier,0.6099,0.5000,0.3789,1.0000,0.0000,0.0000,0.5000
4,Logistic Regression,0.6099,0.5000,0.3789,1.0000,0.0000,0.0000,0.4774


## 5. Select on CV, then fit on the full training partition
No model is chosen based on test-set results.

In [6]:
selected_name=str(comparison.iloc[0]['model'])
selected=candidates[selected_name]
selected.fit(X_train,y_train)
print('Selected (training-only CV):',selected_name)
print('Pipeline:',selected)

Selected (training-only CV): Random Forest
Pipeline: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 RandomForestClassifier(min_samples_leaf=3, n_estimators=160,
                                        n_jobs=1, random_state=42))])


## 6. One-time evaluation on the untouched held-out partition
In the confusion matrix, rows are **actual** classes [0,1] and columns are **predicted** classes [0,1]. Cell [0,1] is the false-safe error count.

In [7]:
metrics,test_predictions=evaluate_model(selected,X_test,y_test)
for key in ['accuracy','balanced_accuracy','macro_f1','precision_potable','recall_potable','recall_not_potable','roc_auc','false_safe_count','false_safe_rate']:
    print(f'{key}: {metrics[key]}')
display(pd.DataFrame(metrics['confusion_matrix'],index=['Actual 0','Actual 1'],columns=['Predicted 0','Predicted 1']))
print('Important: these are retrospective held-out dataset metrics, NOT real-world water-safety validation.')

accuracy: 0.6615853658536586
balanced_accuracy: 0.5973437500000001
macro_f1: 0.5875055232656945
precision_potable: 0.639344262295082
recall_potable: 0.3046875
recall_not_potable: 0.89
roc_auc: 0.6553125000000001
false_safe_count: 44
false_safe_rate: 0.11


,Predicted 0,Predicted 1
Actual 0,356,44
Actual 1,178,78


Important: these are retrospective held-out dataset metrics, NOT real-world water-safety validation.


## 7. Export the complete fitted pipeline, scores and representative held-out records
These are the same artifacts consumed by the interactive website. Serialized `.joblib` files should only be loaded from a trusted repository/environment.

In [8]:
details=save_outputs(df,X_train,X_test,y_train,y_test,comparison,selected_name,selected,metrics,test_predictions)
print('Saved model:',project_root/'models'/'trained_pipeline.joblib')
print('Saved CV table:',project_root/'results'/'model_comparison.csv')
print('Saved final metrics:',project_root/'models'/'model_metrics.json')

Saved model: /mnt/data/Water_Quality_ML_First/models/trained_pipeline.joblib
Saved CV table: /mnt/data/Water_Quality_ML_First/results/model_comparison.csv
Saved final metrics: /mnt/data/Water_Quality_ML_First/models/model_metrics.json


## 8. Live inference example using the saved Pipeline
This is a **real model prediction** for a held-out dataset record. It is not a drinking-water safety test. Allowable missing values are imputed using training-only medians.

In [9]:
from ml.predict import artifacts,predict_one
artifacts.cache_clear()  # refresh serialized model if this notebook was re-run
sample={f:(None if pd.isna(X_test.iloc[0][f]) else float(X_test.iloc[0][f])) for f in FEATURES}
result=predict_one(sample)
display({'sample_id':int(X_test.index[0]),'actual_dataset_label':int(y_test.iloc[0]),'model_prediction':result['predicted_label'],'uncalibrated_model_score_for_class_1':result['class_1_model_score_uncalibrated'],'warnings':result['input_warnings']})

{'sample_id': 664,
 'actual_dataset_label': 1,
 'model_prediction': 'Not Potable (0) — dataset label only',
 'uncalibrated_model_score_for_class_1': 0.2873040674603174,
 'warnings': ['ph: missing value replaced using the training-set median (not a measured value).',
  'Sulfate: missing value replaced using the training-set median (not a measured value).']}

## 9. Scientific interpretation / limits
- **Model selection:** uses 5-fold CV **on training only**. The test is evaluated once at the end.
- **False-safe errors:** a prediction of class 1 for an actual class 0; these are especially concerning.
- **Probabilities:** the estimator scores are **not calibrated** and must not be presented as real water-safety probabilities.
- **Deployment:** expects the same **nine** laboratory-defined features; a low-cost ESP32 sensor kit does *not* provide all nine features.
- **Scope:** educational classification of public dataset labels only. Not a substitute for chemical, microbiological, or regulated laboratory testing.